# Ch.4 — CI/CD Pipelines (Azure Supplement)

This notebook covers Azure-specific CI/CD using **Azure DevOps Pipelines**.

**What you'll learn:**
- Azure DevOps Pipelines (alternative to GitHub Actions)
- Build and push to Azure Container Registry (ACR)
- Deploy to Azure Container Instances (ACI)
- Deploy to Azure Kubernetes Service (AKS)
- Azure Pipelines YAML syntax

**Prerequisites:**
- Azure account with active subscription
- Azure CLI installed (`az` command)
- Azure DevOps organization

⚠️ **Cost warning:** Azure resources incur charges. Use free tier where available and clean up resources after testing.

## Cell 1: Azure Credentials Setup

Configure Azure credentials for CI/CD pipelines.

**Note:** These credentials will be stripped by the pre-push hook. Never commit real credentials.

In [ ]:
import os
import subprocess
import json

# Azure credentials (stripped by pre-push hook)
AZURE_SUBSCRIPTION_ID = ""  # Your Azure subscription ID
AZURE_RESOURCE_GROUP = "rg-devops-demo"
AZURE_LOCATION = "eastus"
AZURE_ACR_NAME = "acrdevopsdemo"  # Must be globally unique
AZURE_ACI_NAME = "aci-flask-app"

# Login to Azure
print("🔐 Logging in to Azure...")
subprocess.run(['az', 'login'], check=True)

# Set active subscription
if AZURE_SUBSCRIPTION_ID:
    subprocess.run(['az', 'account', 'set', '--subscription', AZURE_SUBSCRIPTION_ID], check=True)
    print(f"✅ Active subscription: {AZURE_SUBSCRIPTION_ID}")
else:
    print("⚠️  No subscription ID set. Using default subscription.")

# Get current subscription info
result = subprocess.run(
    ['az', 'account', 'show', '--output', 'json'],
    capture_output=True,
    text=True,
    check=True
)
account_info = json.loads(result.stdout)
print(f"\nActive subscription: {account_info['name']}")
print(f"Subscription ID: {account_info['id']}")

# Create resource group
print(f"\n📦 Creating resource group: {AZURE_RESOURCE_GROUP}")
subprocess.run([
    'az', 'group', 'create',
    '--name', AZURE_RESOURCE_GROUP,
    '--location', AZURE_LOCATION
], check=True)

print(f"✅ Resource group {AZURE_RESOURCE_GROUP} created in {AZURE_LOCATION}")

## Cell 2: Azure DevOps Pipelines (Alternative to GitHub Actions)

Create an Azure DevOps pipeline for CI/CD.

**Azure Pipelines vs GitHub Actions:**

| Feature | GitHub Actions | Azure Pipelines |
|---------|----------------|------------------|
| **Free tier** | 2,000 min/month | 1,800 min/month (private), unlimited (public) |
| **YAML syntax** | `on`, `jobs`, `steps` | `trigger`, `stages`, `jobs`, `steps` |
| **Marketplace** | GitHub Marketplace | Azure DevOps Marketplace |
| **Best for** | GitHub-centric workflows | Azure-centric deployments |

Both use YAML and support similar features (matrix builds, caching, secrets, artifacts).

In [ ]:
# Create Azure Pipelines YAML
azure_pipeline = '''
# azure-pipelines.yml
trigger:
  branches:
    include:
      - main

pool:
  vmImage: 'ubuntu-latest'

variables:
  imageName: 'flask-app'
  acrName: 'acrdevopsdemo'  # Replace with your ACR name

stages:
  - stage: Test
    jobs:
      - job: RunTests
        steps:
          - task: UsePythonVersion@0
            inputs:
              versionSpec: '3.11'
          
          - script: |
              pip install -r requirements.txt
              pytest tests/ -v
            displayName: 'Run unit tests'
  
  - stage: Build
    dependsOn: Test
    condition: succeeded()
    jobs:
      - job: BuildAndPush
        steps:
          - task: Docker@2
            displayName: 'Build and push Docker image'
            inputs:
              containerRegistry: 'ACRServiceConnection'  # Set up in Azure DevOps
              repository: $(imageName)
              command: 'buildAndPush'
              Dockerfile: '**/Dockerfile'
              tags: |
                $(Build.BuildId)
                latest
  
  - stage: Deploy
    dependsOn: Build
    condition: succeeded()
    jobs:
      - deployment: DeployToACI
        environment: 'production'
        strategy:
          runOnce:
            deploy:
              steps:
                - task: AzureCLI@2
                  displayName: 'Deploy to Azure Container Instances'
                  inputs:
                    azureSubscription: 'AzureServiceConnection'
                    scriptType: 'bash'
                    scriptLocation: 'inlineScript'
                    inlineScript: |
                      az container create \
                        --resource-group rg-devops-demo \
                        --name aci-flask-app \
                        --image $(acrName).azurecr.io/$(imageName):$(Build.BuildId) \
                        --cpu 1 --memory 1 \
                        --ports 5000 \
                        --dns-name-label flask-app-demo
'''

with open('azure-pipelines.yml', 'w') as f:
    f.write(azure_pipeline)

print("✅ Azure Pipelines YAML created!")
print("\nKey differences from GitHub Actions:")
print("1. trigger: → on:")
print("2. pool: → runs-on:")
print("3. stages: → jobs: (GitHub Actions has no stages)")
print("4. task: → uses: (Azure uses tasks, GitHub uses actions)")
print("\nTo set up in Azure DevOps:")
print("1. Go to dev.azure.com")
print("2. Create new project")
print("3. Pipelines → New pipeline → Existing YAML file")
print("4. Select azure-pipelines.yml")
print("5. Set up service connections (ACR, Azure subscription)")

## Cell 3: Build and Push to Azure Container Registry (ACR)

Create ACR and push Docker images from CI pipeline.

In [ ]:
# Create Azure Container Registry
print(f"🐳 Creating Azure Container Registry: {AZURE_ACR_NAME}")
subprocess.run([
    'az', 'acr', 'create',
    '--resource-group', AZURE_RESOURCE_GROUP,
    '--name', AZURE_ACR_NAME,
    '--sku', 'Basic',  # Basic = $5/month, 10 GB storage
    '--admin-enabled', 'true'
], check=True)

print(f"✅ ACR {AZURE_ACR_NAME} created!")

# Get ACR credentials
result = subprocess.run([
    'az', 'acr', 'credential', 'show',
    '--name', AZURE_ACR_NAME,
    '--output', 'json'
], capture_output=True, text=True, check=True)
acr_creds = json.loads(result.stdout)

print("\n🔑 ACR credentials (store as pipeline secrets):")
print(f"Username: {acr_creds['username']}")
print(f"Password: {acr_creds['passwords'][0]['value'][:10]}...")

# Login to ACR
subprocess.run(['az', 'acr', 'login', '--name', AZURE_ACR_NAME], check=True)

# Build and push sample image
image_tag = f"{AZURE_ACR_NAME}.azurecr.io/flask-app:v1"

# Create sample Dockerfile if not exists
if not os.path.exists('Dockerfile'):
    with open('Dockerfile', 'w') as f:
        f.write('''
FROM python:3.11-slim
WORKDIR /app
RUN pip install flask
COPY . .
CMD ["python", "-c", "print('Hello from ACR!')"]
''')

print(f"\n🏗️  Building image: {image_tag}")
subprocess.run(['docker', 'build', '-t', image_tag, '.'], check=True)

print(f"📤 Pushing to ACR: {image_tag}")
subprocess.run(['docker', 'push', image_tag], check=True)

print(f"\n✅ Image pushed to ACR!")
print(f"\nTo use in pipeline:")
print(f"  image: {AZURE_ACR_NAME}.azurecr.io/flask-app:latest")

## Cell 4: Deploy to Azure Container Instances (ACI) or AKS

Deploy containerized app to Azure using ACI (simple) or AKS (production).

In [ ]:
# Option 1: Deploy to Azure Container Instances (ACI)
# Fast, serverless, good for dev/test

print("🚀 Deploying to Azure Container Instances...")

aci_result = subprocess.run([
    'az', 'container', 'create',
    '--resource-group', AZURE_RESOURCE_GROUP,
    '--name', AZURE_ACI_NAME,
    '--image', f"{AZURE_ACR_NAME}.azurecr.io/flask-app:v1",
    '--cpu', '1',
    '--memory', '1',
    '--registry-login-server', f"{AZURE_ACR_NAME}.azurecr.io",
    '--registry-username', acr_creds['username'],
    '--registry-password', acr_creds['passwords'][0]['value'],
    '--ip-address', 'Public',
    '--ports', '5000',
    '--dns-name-label', f"flask-app-{AZURE_ACR_NAME}",
    '--output', 'json'
], capture_output=True, text=True, check=True)

aci_info = json.loads(aci_result.stdout)
fqdn = aci_info['ipAddress']['fqdn']

print(f"✅ Deployed to ACI!")
print(f"URL: http://{fqdn}:5000")
print(f"\nCheck status:")
print(f"  az container show -g {AZURE_RESOURCE_GROUP} -n {AZURE_ACI_NAME}")

# Option 2: Deploy to AKS (commented out, more complex)
print("\n" + "="*60)
print("To deploy to AKS instead (production setup):")
print("="*60)
print("""
# Create AKS cluster
az aks create \
  --resource-group rg-devops-demo \
  --name aks-cluster \
  --node-count 2 \
  --enable-addons monitoring \
  --attach-acr acrdevopsdemo

# Get credentials
az aks get-credentials --resource-group rg-devops-demo --name aks-cluster

# Deploy using kubectl (same as Kind)
kubectl apply -f k8s/deployment.yaml
""")

print("\n💰 Cost comparison:")
print("- ACI: ~$0.0000125/vCPU-second (~$10/month for 1 vCPU)")
print("- AKS: Free control plane + VM costs (~$70/month for 2 B2s nodes)")
print("\n⚠️  Remember to delete resources to avoid charges!")

## Cell 5: Azure Pipelines YAML Syntax

Comprehensive reference for Azure Pipelines syntax.

In [ ]:
# Complete Azure Pipelines reference
reference = '''
# =====================================
# Azure Pipelines YAML Syntax Reference
# =====================================

# Triggers
trigger:
  branches:
    include:
      - main
      - develop
  paths:
    include:
      - src/**
    exclude:
      - docs/**

# PR triggers
pr:
  branches:
    include:
      - main

# Scheduled triggers
schedules:
  - cron: "0 0 * * *"  # Daily at midnight
    displayName: Nightly build
    branches:
      include:
        - main

# Variables
variables:
  - name: buildConfiguration
    value: 'Release'
  - group: prod-secrets  # Variable group from Azure DevOps

# Agent pool
pool:
  vmImage: 'ubuntu-latest'  # or windows-latest, macos-latest

# Stages (groups jobs)
stages:
  - stage: Build
    displayName: 'Build stage'
    jobs:
      - job: BuildJob
        steps: [...]

# Jobs
jobs:
  - job: Test
    displayName: 'Run tests'
    timeoutInMinutes: 30
    strategy:
      matrix:
        Python39:
          python.version: '3.9'
        Python311:
          python.version: '3.11'
    steps: [...]

# Steps
steps:
  # Script (bash)
  - script: |
      echo "Building..."
      python build.py
    displayName: 'Build app'
  
  # Task (pre-built action)
  - task: UsePythonVersion@0
    inputs:
      versionSpec: '3.11'
  
  # Checkout (default, but can customize)
  - checkout: self
    clean: true
  
  # Download artifact
  - download: current
    artifact: drop
  
  # Publish artifact
  - publish: $(Build.ArtifactStagingDirectory)
    artifact: drop

# Deployment job
- deployment: DeployWeb
  environment: production
  strategy:
    runOnce:
      deploy:
        steps:
          - script: echo Deploying

# Conditions
- job: DeployProd
  condition: and(succeeded(), eq(variables['Build.SourceBranch'], 'refs/heads/main'))
  steps: [...]

# Predefined variables
# $(Build.BuildId) - Unique build ID
# $(Build.SourceBranch) - refs/heads/main
# $(Build.Repository.Name) - Repo name
# $(System.DefaultWorkingDirectory) - Source directory
# $(Agent.BuildDirectory) - Build output directory
'''

print(reference)

print("\n" + "="*60)
print("Common Azure Pipelines tasks:")
print("="*60)
print("""
UsePythonVersion@0       - Install Python
Docker@2                 - Build/push Docker images
AzureCLI@2              - Run Azure CLI commands
Kubernetes@1            - Deploy to Kubernetes
PublishTestResults@2    - Publish test results
PublishCodeCoverageResults@1 - Publish coverage
DownloadBuildArtifacts@1 - Download artifacts
PublishBuildArtifacts@1  - Upload artifacts
""")

print("\n📚 Full reference:")
print("https://docs.microsoft.com/azure/devops/pipelines/yaml-schema")

## Cleanup

**Important:** Delete Azure resources to avoid charges.

In [ ]:
print("🗑️  To delete all resources:")
print(f"\naz group delete --name {AZURE_RESOURCE_GROUP} --yes --no-wait")
print("\nThis deletes:")
print("- ACR (Azure Container Registry)")
print("- ACI (Azure Container Instances)")
print("- All other resources in the group")